# Figure 4 (corrected): PDFs of the near-bottom vertical velocity, moorings vs CROCO

Replots `plot_figure4.py` from the same files (`RREX_<mooring>_CURR_hourly.nc`, `rrex_dD_10km.nc`, `rrex_bottom_w_mooring_notracer.nc`), with these corrections.

| `plot_figure4.py` | here |
|---|---|
| IRE quality flag applied to the IRW arrays (`u_irw[uirw_qc>2]`) | each mooring is masked with its own flags |
| deepest bin taken as the last column `[:, -1]` | deepest bin = largest median depth (checked against the last column) |
| `np.median` / `np.std` on arrays that may contain NaN | `nan`-aware statistics |
| PDF normalised by `len(data)` (counts NaN) | normalised by the number of valid samples |
| legend says "mean" but a **median** is drawn | mean ± std (solid / dashed) and the median (dotted) are both drawn and labelled |
| panel titles with a hard-coded `hab` (100, 93, 116, ...) and legend/depth labels (`-2060 m`, `label_croco`) from an older extraction | height above bottom **computed** as `depth_ref - z_sensor`; the hab list used in the old CROCO file is compared and the mismatches (ICM, ICE) are flagged |
| fixed axes / texts for one panel only | same axes, legend and units on all panels |

**Same height above bottom.** The mooring bin is at `hab_obs = depth_ref - z_sensor`, where `depth_ref` is the surveyed water depth at the mooring (the values of `depth_cv` in the old script) and `z_sensor` the median depth of the deepest bin. The model has to be sampled at the same height above *its own* sea floor, `z = -h_model + hab_obs` (the model floor differs from the real one by tens of metres, so equal hab is not equal depth). `extract_bottom_w_moorings_croco_hab.py` does that and writes `rrex_w_moorings_hab.nc`; this notebook uses it when it exists and otherwise falls back to the old file.

**What the mooring gives.** The deepest current-meter bin of each mooring, converted to the bottom-slope vertical velocity $w_u=-\mathbf{u}\cdot\nabla D$ [m day$^{-1}$] with the 10 km-smoothed depth gradient (`dDdx`, `dDdy`). **What CROCO gives.** The same quantity (`wub*`) or the true vertical velocity (`wb*`) interpolated in the model at the extraction depth `Dd = depth_CROCO - hab`, from the file `rrex_bottom_w_mooring_notracer.nc`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter, NullFormatter
from netCDF4 import Dataset
from pathlib import Path

# ---------------- configuration ----------------
DATA_DIR  = Path('./data')                      # folder with the three kinds of files
FILE_D    = DATA_DIR / 'rrex_dD_10km.nc'        # only used as a fallback if the new extraction file has no SRTM15 slopes
OBS_SLOPE_KM   = 10.                            # smoothing length [km] of the OBSERVED slope (SRTM15), among slope_length_km of FILE_HAB
MODEL_SLOPE_KM = 10.                            # smoothing length [km] of the MODEL slope (0 = grid scale), chosen independently of the obs
FILE_CROCO= DATA_DIR / 'rrex_bottom_w_mooring_notracer.nc'   # old extraction (hand-typed hab list)
FILE_HAB  = DATA_DIR / 'rrex_w_moorings_hab.nc'              # new extraction at hab_obs (extract_bottom_w_moorings_croco_hab.py)
MOORINGS  = ['IRW', 'IRM', 'IRE', 'RRT', 'ICW', 'ICM', 'ICE']    # index in dDdx, wb*, wub* = position in this list

# model quantities to plot (the first is the main one; the others are drawn on top in other colours).  New file:
#   'wu_L' : -u.grad(H smoothed over MODEL_SLOPE_KM)     'wu' : -u.grad(H), grid-scale slope     'w' : true model vertical velocity
# Old file (no new extraction): 'wu_L'/'wu' -> 'wub', 'w' -> 'wb'.
MODEL_SERIES = ['wu_L', 'w']                                  # keep both: figure 1 uses 'wu_L' (or 'w' if MOORING_VAR='WCUR'), figure 2 uses 'wu_L' and 'w'
MOORING_VAR = 'wu'      # 'wu' : -u.grad(D) from the deepest current meter ; 'WCUR': measured vertical velocity
QC_MAX      = 2         # keep samples with QC flag <= QC_MAX (1 = good, 2 = probably good)

# surveyed water depth at the mooring [m] (depth_cv of the old extraction script): reference for the observed hab
DEPTH_REF   = np.array([2160., 1570., 1580., 1528., 2162., 2151., 2379.])
# hab list typed by hand in the OLD extraction (what the old file was sampled at)
HAB_EXTRACT = np.array([100, 93, 116, 81, 57, 46, 282])

LOG_Y = True            # True: logarithmic pdf axis (shows the tails); False: linear axis (figures saved without the '_log' suffix)
YMIN, YMAX = 5e-5, 0.1   # y limits of the log axis, in probability density [(m/day)^-1]
YLIM_LIN = None          # y limits of the linear axis, e.g. (0, 0.03); None = automatic

dw  = 8.                                            # bin width [m/day]
w_e = np.arange(-750, 750 + dw, dw)
XLIM = (-410, 410)
SEC_PER_DAY = 86400.
fs = 17                                             # font size of labels and titles
PANEL_W, PANEL_H = 18., 2.7                         # width of the figure and height of one pdf panel [inch]: wide panels

In [ ]:
def masked_to_nan(a):
    """netCDF masked array -> float array with NaN for masked / fill values."""
    a = np.ma.filled(np.ma.asarray(a).astype(float), np.nan)
    a[np.abs(a) > 1e5] = np.nan                  # 999999 fill values
    return a

def read_mooring(name, i_grad, grad):
    ds = Dataset(DATA_DIR / ('RREX_%s_CURR_hourly.nc' % name))
    depth = masked_to_nan(ds['DEPTH'][:])
    zmed  = np.nanmedian(depth, axis=0)
    k     = int(np.nanargmax(zmed))                   # deepest bin
    out   = dict(name=name, k=k, nlev=depth.shape[1], k_is_last=(k == depth.shape[1] - 1), z=zmed[k])
    def get(var):
        x = masked_to_nan(ds[var][:, k])
        if var + '_QC' in ds.variables:
            q = np.ma.filled(ds[var + '_QC'][:, k], 9)
            x[q > QC_MAX] = np.nan
        return x
    u, v = get('UCUR'), get('VCUR')
    out['wu']   = -(u * grad['dDdx'][i_grad] + v * grad['dDdy'][i_grad]) * SEC_PER_DAY   # D > 0: bottom at z = -D
    out['WCUR'] = get('WCUR') * SEC_PER_DAY
    ds.close()
    return out

# observed bottom slope at the moorings: SRTM15 slope smoothed over OBS_SLOPE_KM, computed by the extraction script
if FILE_HAB.exists() and 'dDdx_topo15' in Dataset(FILE_HAB).variables:
    with Dataset(FILE_HAB) as nc:
        L = np.asarray(nc['slope_length_km'][:], float)
        k = int(np.argmin(abs(L - OBS_SLOPE_KM)))
        grad = {'dDdx': np.asarray(nc['dDdx_topo15'][k], float), 'dDdy': np.asarray(nc['dDdy_topo15'][k], float),
                'D': np.asarray(nc['D_topo15'][:], float)}
    print('observed slope: SRTM15 smoothed over %g km (from %s)' % (L[k], FILE_HAB.name))
else:
    with Dataset(FILE_D) as nd:                                 # fallback: precomputed slopes (method not documented)
        grad = {k: np.asarray(nd[k][:], float) for k in ('D', 'dDdx', 'dDdy')}
    print('observed slope: %s (no SRTM15 slopes found)' % FILE_D.name)
obs = [read_mooring(m, i, grad) for i, m in enumerate(MOORINGS)]

USE_NEW = FILE_HAB.exists()
mod, mod_label = {}, {}                                          # mod[name] = list over moorings of the model series
if USE_NEW:
    with Dataset(FILE_HAB) as nc:
        Lm = np.asarray(nc['slope_length_km'][:], float) if 'slope_length_km' in nc.variables else np.array([])
        for name in MODEL_SERIES:
            if name == 'w':
                arr, mod_label[name] = nc['w_obs_hab'][:], r'CROCO: $w$'
            elif name == 'wu' or (name == 'wu_L' and MODEL_SLOPE_KM == 0 and 'wu_L_obs_hab' not in nc.variables):
                arr, mod_label[name] = nc['wu_obs_hab'][:], r'CROCO: $-\mathbf{u}\cdot\nabla H$ (grid scale)'
            elif 'wu_L_obs_hab' in nc.variables:
                kL = int(np.argmin(abs(Lm - MODEL_SLOPE_KM)))
                assert abs(Lm[kL] - MODEL_SLOPE_KM) < 1e-6, 'MODEL_SLOPE_KM=%g not in %s' % (MODEL_SLOPE_KM, Lm)
                arr, mod_label[name] = nc['wu_L_obs_hab'][:, :, kL], r'CROCO: $-\mathbf{u}\cdot\nabla H$'
            else:                                                # file from the previous version of the extraction
                print('wu_L_obs_hab not in %s: using wu_s (smoothing %g km)' % (FILE_HAB.name, nc.smooth_km))
                arr, mod_label[name] = nc['wu_s_obs_hab'][:], r'CROCO: $-\mathbf{u}\cdot\nabla H$'
            arr = np.asarray(np.ma.filled(np.ma.asarray(arr).astype(float), np.nan), float)    # (time, mooring)
            mod[name] = [arr[:, i] for i in range(len(MOORINGS))]
        hab_model = np.asarray(nc['hab_obs'][:], float)           # hab at which the model was sampled
        h_model   = np.asarray(nc['h'][:], float)
    print('model file: %s (sampled at hab_obs), series %s' % (FILE_HAB.name, ', '.join(MODEL_SERIES)))
else:
    with Dataset(FILE_CROCO) as nc:
        for name in MODEL_SERIES:
            old = 'wb' if name == 'w' else 'wub'
            mod[name] = [np.asarray(nc['%s%d' % (old, i)][:], float) for i in range(len(MOORINGS))]
            mod_label[name] = r'CROCO: $w$' if name == 'w' else r'CROCO: $-\mathbf{u}\cdot\nabla H$'
    hab_model = HAB_EXTRACT.astype(float)
    h_model   = None
    print('%s not found: using the OLD file %s, sampled at the hand-typed hab list' % (FILE_HAB.name, FILE_CROCO.name))


In [ ]:
def clean(x): return x[np.isfinite(x)]
print('%-4s %8s %10s | %8s | %10s %9s' % ('', 'z_sensor', 'depth_ref', 'hab_obs', 'hab model', 'model-obs'))
for i, o in enumerate(obs):
    o['hab'] = DEPTH_REF[i] - o['z']                  # height of the deepest bin above the surveyed bottom
    d = hab_model[i] - o['hab']
    print('%-4s %8.1f %10.0f | %8.1f | %10.1f %9.1f' % (o['name'], o['z'], DEPTH_REF[i], o['hab'], hab_model[i], d))
print()
print('hab_obs   = depth_ref - z_sensor : height of the deepest current meter above the surveyed bottom')
print('hab model = height above the MODEL floor at which the model series was sampled')
bad = [o['name'] for i, o in enumerate(obs) if abs(hab_model[i] - o['hab']) > 5]
if bad:
    print('\n!! model and mooring are NOT at the same hab (> 5 m apart) for: %s' % ', '.join(bad))
    print('   -> run extract_bottom_w_moorings_croco_hab.py, which samples the model at hab_obs')
else:
    print('\nmodel and mooring are at the same height above bottom (within 5 m) at all moorings')

In [ ]:
stats = []
print('\n%-4s %-8s %6s | %26s' % ('', 'series', 'N', 'mean / median / std [m/day]'))
for i, o in enumerate(obs):
    xo = clean(o[MOORING_VAR])
    s = dict(xo=xo, mo=xo.mean(), do=np.median(xo), so=xo.std(), model={})
    print('%-4s %-8s %6d | %8.1f %8.1f %8.1f' % (o['name'], 'mooring', len(xo), s['mo'], s['do'], s['so']))
    for name in MODEL_SERIES:
        xm = clean(mod[name][i])
        s['model'][name] = dict(x=xm, m=xm.mean(), d=np.median(xm), s=xm.std())
        print('%-4s %-8s %6d | %8.1f %8.1f %8.1f' % ('', name, len(xm), xm.mean(), np.median(xm), xm.std()))
    stats.append(s)


In [ ]:
label_w = {'wu': r'$-\mathbf{u}\cdot\nabla H$ [m day$^{-1}$]', 'WCUR': r'$w$ (ADCP) [m day$^{-1}$]'}[MOORING_VAR]
letters = 'abcdefg'
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
c_obs, c_obs_stat = 'm', 'r'                                    # mooring: histogram, statistics lines (colours of plot_figure4.py)
SERIES_STYLE = {'wu_L': ('royalblue', 'k'), 'wu': ('royalblue', 'dimgray'), 'w': ('teal', 'darkgreen')}   # model: (histogram, statistics lines); royalblue = -u.grad(H) and teal = w as in plot_figure4.py
wt = lambda x: np.ones(len(x)) / (len(x) * dw)                  # probability density: integrates to 1 over w [(m/day)^-1]


def plot_pdfs(with_obs, series, xlabel, fname):
    """One pdf panel per mooring: the mooring (with_obs) and/or the model series in `series`."""
    fig, axes = plt.subplots(len(MOORINGS), 1, figsize=(PANEL_W, PANEL_H * len(MOORINGS)), sharex=True)
    for i, (ax, o, s) in enumerate(zip(axes, obs, stats)):
        stat_list = []
        if with_obs:
            ax.hist(s['xo'], bins=w_e, weights=wt(s['xo']), color=c_obs, alpha=0.5, label='Mooring (deepest bin)')
            stat_list.append((c_obs_stat, 'mooring', s['mo'], s['do'], s['so'], c_obs, 'Mooring (deepest bin)'))
        for name in series:
            m = s['model'][name]
            ax.hist(m['x'], bins=w_e, weights=wt(m['x']), color=SERIES_STYLE[name][0], alpha=0.5, label=mod_label[name])
            stat_list.append((SERIES_STYLE[name][1], mod_label[name].replace('CROCO: ', ''), m['m'], m['d'], m['s'], SERIES_STYLE[name][0], mod_label[name]))
        for col, who, m_, d_, s_, hc, hl in stat_list:
            ax.axvline(m_, color=col, lw=2)
            ax.axvline(m_ + s_, color=col, lw=2, ls='--')
            ax.axvline(m_ - s_, color=col, lw=2, ls='--')
        ax.set_xlim(*XLIM)
        if LOG_Y:
            ax.set_yscale('log')
            ax.set_ylim(YMIN, YMAX)
            ax.yaxis.set_minor_formatter(NullFormatter())
        elif YLIM_LIN is not None:
            ax.set_ylim(*YLIM_LIN)
        ax.set_ylabel(r'pdf [(m day$^{-1}$)$^{-1}$]', fontsize=fs); ax.tick_params(labelsize=fs - 2)
        ax.grid(True, which='major', alpha=0.5)
        ax.grid(True, which='minor', axis='y', alpha=0.2)
        ax.set_axisbelow(True)
        t_ = '%s) %s: sensor at %.0f m, hab = %.0f m' % (letters[i], o['name'], o['z'], o['hab'])
        if abs(hab_model[i] - o['hab']) > 5:
            t_ += '  [model sampled at hab = %.0f m!]' % hab_model[i]
        ax.set_title(t_, fontsize=fs, color='r' if abs(hab_model[i] - o['hab']) > 5 else 'k')
    axes[-1].set_xlabel(xlabel, fontsize=fs)
    # legend: one row per data set (mooring / each model series): histogram | mean | mean +- std | median.
    # matplotlib fills legend columns first, so the handles are given column by column.
    rows = [[Patch(color=hc, alpha=0.5, label=hl), Line2D([], [], color=col, lw=2, label='mean'),
             Line2D([], [], color=col, lw=2, ls='--', label=r'mean $\pm$ std')]
            for col, who, m_, d_, s_, hc, hl in stat_list]
    handles = [rows[r][c] for c in range(3) for r in range(len(rows))]
    fig.legend(handles=handles, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.0), fontsize=fs - 2, columnspacing=2.5)
    fig.tight_layout(rect=(0, 0, 1, 0.965))
    fig.savefig(fname + ('_log.png' if LOG_Y else '.png'), dpi=180, bbox_inches='tight')
    plt.show()


# 1) mooring vs model, same quantity: w_u = -u.grad(depth)  (observed slope length OBS_SLOPE_KM, model slope length MODEL_SLOPE_KM)
if MOORING_VAR == 'wu':
    plot_pdfs(True, ['wu_L'], label_w, 'figure4_corrected_obs_vs_model')
else:                                       # MOORING_VAR = 'WCUR': compare the measured w with the model true w
    plot_pdfs(True, ['w'], label_w, 'figure4_corrected_obs_vs_model')


In [ ]:
# 2) model only: true vertical velocity w  vs  w_u = -u.grad(H)  (the part of w that follows the bottom slope)
plot_pdfs(False, ['wu_L', 'w'], r'model vertical velocity [m day$^{-1}$]', 'figure4_corrected_model_w_vs_wu')
